# Práctica A: ¿Puede la IA predecir el clima? Pronóstico Multi-modelo SENAMHI

Este notebook implementa el caso de estudio **Caso A** de la charla *¿Puede la IA predecir el futuro? Lecciones de forecasting desde el mundo real*.

**Estación:** Punchana (Iquitos, Loreto, Perú - Código SENAMHI 103015)
**Target:** Temperatura máxima diaria ($T_{\text{max}}$)
**Validación:** Holdout temporal estricto de los últimos 90 días (sin fuga de datos).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
import xgboost as xgb
from statsmodels.tsa.arima.model import ARIMA
import torch
import torch.nn as nn

# Cargar serie diaria limpia (2019-2025)
df = pd.read_csv('../data/senamhi_diario.csv', parse_dates=['fecha']).set_index('fecha')
print(f'Total observaciones: {len(df)} días')
df[['tmax', 'precip']].head()

## 1. Ingeniería de Características (Lags y Calendario)
En series temporales con ML tabular, transformamos la secuencia en una matriz supervisada respetando la flecha del tiempo.

In [ ]:
df_feat = df.copy()
for lag in range(1, 15):
    df_feat[f'lag_{lag}'] = df_feat['tmax'].shift(lag)

df_feat['roll_mean_7'] = df_feat['tmax'].shift(1).rolling(7).mean()
df_feat['roll_std_7'] = df_feat['tmax'].shift(1).rolling(7).std()
df_feat['precip_lag1'] = df_feat['precip'].shift(1)
df_feat['sin_dia'] = np.sin(2 * np.pi * df_feat['dia_anio'] / 365.25)
df_feat['cos_dia'] = np.cos(2 * np.pi * df_feat['dia_anio'] / 365.25)
df_feat = df_feat.dropna()

# Split temporal últimos 90 días para holdout
holdout = 90
train_df = df_feat.iloc[:-holdout]
test_df = df_feat.iloc[-holdout:]

features = [c for c in df_feat.columns if c.startswith('lag_') or c.startswith('roll_') or c in ['precip_lag1', 'sin_dia', 'cos_dia', 'mes']]
X_train, y_train = train_df[features], train_df['tmax']
X_test, y_test = test_df[features], test_df['tmax']
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

## 2. Ajuste de Modelos (Naive, ARIMA, Random Forest, ExtraTrees, XGBoost, LSTM)

In [ ]:
models = {
    'Naive Persistencia': y_test.shift(1).bfill().values,
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42).fit(X_train, y_train).predict(X_test),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=100, max_depth=8, random_state=42).fit(X_train, y_train).predict(X_test),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42).fit(X_train, y_train).predict(X_test)
}

results = []
for name, preds in models.items():
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    results.append({'Modelo': name, 'RMSE (°C)': round(rmse, 3), 'MAE (°C)': round(mae, 3)})

pd.DataFrame(results)